# RotorPy Simulation Test

This notebook creates a basic RotorPy simulation to test out the most important aspect of the library.

## Instantiation

First, all necessary instances for the simulation are created.

In [ ]:
from rotorpy.sensors.imu import Imu

SIM_HZ_RATE = 1000

In [ ]:
import numpy as np
from rotorpy.controllers.quadrotor_control import SE3Control
from rotorpy.environments import Environment
from rotorpy.trajectories.circular_traj import ThreeDCircularTraj
from rotorpy.vehicles.crazyflie_params import quad_params
from rotorpy.vehicles.multirotor import Multirotor
from rotorpy.world import World

crazyflie = Multirotor(quad_params)
world: World = World.empty(extents=[-3, 3, -3, 3, 0, 5])
trajectory = ThreeDCircularTraj(center=np.array([0, 0, 2]), radius=np.array([2, 2, 0]), freq=np.array([0.2, 0.2, 0.2]))
imu = Imu(accelerometer_params={'initial_bias': np.array([0., 0., 0.]),  # m/s^2
                                'noise_density': np.zeros(3, ),  # m/s^2 / sqrt(Hz)
                                'random_walk': np.zeros(3, )  # m/s^2 * sqrt(Hz)
                                },
          gyroscope_params={'initial_bias': np.array([0., 0., 0.]),  # m/s^2
                            'noise_density': np.zeros(3, ),  # rad/s / sqrt(Hz)
                            'random_walk': np.zeros(3, )  # rad/s * sqrt(Hz)
                            }, )

sim_instance = Environment(vehicle=crazyflie,
                           controller=SE3Control(quad_params),
                           trajectory=trajectory,
                           world=world,
                           wind_profile=None,
                           sim_rate=SIM_HZ_RATE,
                           safety_margin=0.25)

## Execution

This part runs the actual simulation of the crazyflie.

In [ ]:

# initial state
x0 = {'x': np.array([2, 0, 2]),
      'v': np.zeros(3, ),
      'q': np.array([0, 0, 0, 1]),  # [i,j,k,w]
      'w': np.zeros(3, ),
      'wind': np.array([0, 0, 0]),  # Since wind is handled elsewhere, this value is overwritten
      'rotor_speeds': np.array([1788.53, 1788.53, 1788.53, 1788.53])}
sim_instance.vehicle.initial_state = x0

In [ ]:
results = sim_instance.run(t_final=10,  # The maximum duration of the environment in seconds
                           use_mocap=False,
                           terminate=False,
                           plot=True,
                           plot_mocap=False,
                           plot_estimator=False,
                           plot_imu=True,
                           animate_bool=False,
                           animate_wind=False,
                           verbose=False,
                           fname=None
                           )


## Evaluation

This part will explore the data generated by the simulation.

In [ ]:
from rotorpy.utils.postprocessing import unpack_sim_data

df = unpack_sim_data(results)

In [ ]:
df.info()

In [ ]:
df.head(n=10)

In [ ]:
results["state"].keys()

In [ ]:
results["imu_measurements"]

In [ ]:
results["imu_gt"]